# Fine-Tuning Whisper on Nepali (LoRA / PEFT)

This notebook fine-tunes OpenAI's Whisper model on **Nepali speech** using the
[Mozilla Common Voice](https://commonvoice.mozilla.org/) Nepali dataset, with
**LoRA (PEFT)** so it stays light enough to train on a free-tier GPU.

**Important — run this on a GPU, not your laptop:**
- Training (unlike inference) needs real GPU VRAM. On an 8GB Apple Silicon Mac
  this will be extremely slow or fail outright.
- Use **Google Colab** (Runtime → Change runtime type → T4 GPU, free tier) or
  any other CUDA GPU environment.
- Once trained, the resulting model is small and runs fine for *inference* back
  on your Mac (same as the `Dragneel/whisper-small-nepali` checkpoint you used
  earlier).

**Why LoRA instead of full fine-tuning:**
Full fine-tuning updates all ~244M parameters of Whisper-small and needs a lot
of VRAM. LoRA freezes the base model and trains small adapter layers instead —
far less memory, much faster, and produces a small adapter file (a few MB)
instead of a full new multi-GB model.

## 1. Install dependencies

In [ ]:
%pip install -q transformers datasets accelerate evaluate jiwer peft bitsandbytes librosa soundfile

## 2. Log in to Hugging Face

Needed to download Common Voice (some configs require accepting terms on the
dataset page first — visit its Hugging Face page once and click "Agree") and,
optionally, to push your fine-tuned model back to the Hub at the end.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Load the Nepali Common Voice dataset

In [ ]:
from datasets import load_dataset, Audio, DatasetDict

LANG_CODE = "ne-NP"  # Nepali

common_voice = DatasetDict()
common_voice["train"] = load_dataset(
    "mozilla-foundation/common_voice_16_1", LANG_CODE, split="train+validation"
)
common_voice["test"] = load_dataset(
    "mozilla-foundation/common_voice_16_1", LANG_CODE, split="test"
)

# keep only the columns we need
common_voice = common_voice.remove_columns(
    [c for c in common_voice["train"].column_names if c not in ("audio", "sentence")]
)

# Whisper expects 16kHz audio
common_voice = common_voice.cast_column("audio", Audio(sampling_rate=16000))

print(common_voice)
print("Example:", common_voice["train"][0]["sentence"])

If `common_voice_16_1` isn't available or is too large for a quick run, swap in
an older/smaller config such as `mozilla-foundation/common_voice_11_0` — same
code, just change the dataset name above.

## 4. Load the processor and preprocess the data

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

BASE_MODEL = "openai/whisper-small"  # small = best fit for a free Colab GPU

feature_extractor = WhisperFeatureExtractor.from_pretrained(BASE_MODEL)
tokenizer = WhisperTokenizer.from_pretrained(BASE_MODEL, language="Nepali", task="transcribe")
processor = WhisperProcessor.from_pretrained(BASE_MODEL, language="Nepali", task="transcribe")

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

common_voice = common_voice.map(
    prepare_dataset, remove_columns=common_voice["train"].column_names, num_proc=1
)

`num_proc=1` is deliberate — audio decoding with multiple processes is a common
source of crashes on Colab. Raise it only if you know your environment handles it.

## 5. Data collator (pads audio features and text labels separately)

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # strip the BOS token if the tokenizer added one (it's re-added during generation)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

## 6. Load the base model and wrap it with LoRA

In [ ]:
from transformers import WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model

model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL, load_in_8bit=True, device_map="auto")
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

`load_in_8bit=True` uses `bitsandbytes` to further cut memory — this combo
(8-bit base model + LoRA adapters) is what makes fine-tuning Whisper feasible
on a free T4. You should see only a small fraction of parameters listed as
trainable (typically under 1%).

## 7. Training arguments and Trainer

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="whisper-small-ne-lora",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-3,
    warmup_steps=50,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    per_device_eval_batch_size=8,
    generation_max_length=128,
    logging_steps=25,
    remove_unused_columns=False,  # required for PEFT models
    label_names=["labels"],       # required for PEFT models
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    data_collator=data_collator,
    tokenizer=processor.feature_extractor,
)
model.config.use_cache = False  # silences a harmless warning during training with gradient checkpointing

## 8. Train

On a free Colab T4, expect roughly **1-3 hours** depending on dataset size —
check how many hours of audio your chosen Common Voice split has before
starting, and reduce `num_train_epochs` if you're short on Colab session time.

In [ ]:
trainer.train()

## 9. Save the LoRA adapter

In [ ]:
model.save_pretrained("whisper-small-ne-lora-adapter")
processor.save_pretrained("whisper-small-ne-lora-adapter")

# Optional: push to your own Hugging Face account so you can pull it on your Mac later
# model.push_to_hub("your-username/whisper-small-ne-lora")
# processor.push_to_hub("your-username/whisper-small-ne-lora")

## 10. Run inference with the fine-tuned adapter

This is the code you'll use later — on Colab right after training, or back on
your Mac after downloading/pushing the adapter.

In [ ]:
from peft import PeftModel
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import librosa

base = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
inference_model = PeftModel.from_pretrained(base, "whisper-small-ne-lora-adapter")
inference_processor = WhisperProcessor.from_pretrained("whisper-small-ne-lora-adapter")

audio, sr = librosa.load("./audio/shankar.mp3", sr=16000)
input_features = inference_processor(audio, sampling_rate=16000, return_tensors="pt").input_features

forced_decoder_ids = inference_processor.get_decoder_prompt_ids(language="ne", task="transcribe")
predicted_ids = inference_model.generate(input_features, forced_decoder_ids=forced_decoder_ids)
transcription = inference_processor.batch_decode(predicted_ids, skip_special_tokens=True)

print(transcription[0])

## Notes on your specific use case (movie dialogue)

Common Voice is clean, single-speaker read speech — this fine-tune will
improve general Nepali vocabulary/pronunciation accuracy, but **won't**
specifically teach the model to handle background music or overlapping
dialogue like `shankar.mp3`. If movie-dialogue accuracy is the real goal:
- Test this fine-tuned model against a **clean** Nepali clip first, to confirm
  the fine-tuning itself worked (isolates language accuracy from noise robustness).
- If you have access to any transcribed movie/noisy-speech data, mixing some
  into training (even a small amount) helps more than clean data alone for
  this specific scenario — this is a data problem, not something a bigger
  model alone fixes.